# E-Commerce Sales Data Inspection

Exploratory data analysis of the `data_learn.default.ecommerce_sales_analyst_task` dataset: row count, schema, summary statistics, missing values, duplicates, and unique categorical values.

In [0]:
# Load the dataset
df = spark.table("data_learn.default.ecommerce_sales_analyst_task")

# Row count
total_rows = df.count()
print(f"Total rows: {total_rows:,}")

# Number of columns
num_cols = len(df.columns)
print(f"Number of columns: {num_cols}")

# Schema / data types
print("\n--- Schema (column name → data type) ---")
for field in df.schema.fields:
    print(f"  {field.name:<20} {field.dataType.simpleString()}")

Total rows: 41
Number of columns: 9

--- Schema (column name → data type) ---
  order_id             string
  order_date           date
  region               string
  category             string
  product              string
  quantity             bigint
  unit_price_inr       bigint
  discount_rate        double
  status               string


In [0]:
# Preview first 10 rows
display(df.limit(10))

order_id,order_date,region,category,product,quantity,unit_price_inr,discount_rate,status
O1001,2026-09-01,North,Electronics,Wireless Mouse,2,799,0.05,Delivered
O1002,2026-09-01,South,Home,Desk Lamp,1,1299,0.1,Delivered
O1003,2026-09-02,West,Electronics,USB-C Hub,1,2499,0.0,Delivered
O1004,2026-09-02,East,Fashion,Cotton T-Shirt,3,499,0.15,Delivered
O1005,2026-09-03,North,Home,Storage Box,4,299,0.0,Delivered
O1006,2026-09-03,South,Fashion,Running Shoes,1,2999,0.2,Delivered
O1007,2026-09-04,West,Electronics,Wireless Mouse,1,799,0.0,Cancelled
O1008,2026-09-04,null,Home,Desk Lamp,2,1299,0.05,Delivered
O1009,2026-09-05,North,Fashion,Cotton T-Shirt,2,499,0.1,Returned
O1010,2026-09-05,South,Electronics,USB-C Hub,2,2499,0.1,Delivered


In [0]:
# Summary statistics for numeric columns
print("--- Summary Statistics (numeric columns) ---")
df.summary("count", "min", "25%", "50%", "75%", "max", "mean", "stddev").display()

--- Summary Statistics (numeric columns) ---


summary,order_id,region,category,product,quantity,unit_price_inr,discount_rate,status
count,41,40,41,41,41,41,40,41
min,O1001,East,Electronics,Cotton T-Shirt,0,299,0.0,Cancelled
25%,null,null,null,null,1,499,0.0,null
50%,null,null,null,null,2,799,0.05,null
75%,null,null,null,null,2,2499,0.1,null
max,O1040,West,Home,Wireless Mouse,5,2999,0.25,delivered
mean,null,null,null,null,1.9024390243902438,1359.9756097560976,0.07125,null
stddev,null,null,null,null,1.067822036876475,1006.200290394633,0.06969484769243559,null


In [0]:
# Check for missing (null) values in every column
from pyspark.sql.functions import col, count, when, isnull

print("--- Missing Values per Column ---")
missing_df = df.select([
    count(when(isnull(c), c)).alias(c) for c in df.columns
])
missing_row = missing_df.collect()[0]

for col_name in df.columns:
    null_count = missing_row[col_name]
    pct = (null_count / total_rows) * 100 if total_rows > 0 else 0
    status = "✓ No nulls" if null_count == 0 else f"⚠ {null_count:,} nulls ({pct:.1f}%)"
    print(f"  {col_name:<20} {status}")

--- Missing Values per Column ---
  order_id             ✓ No nulls
  order_date           ✓ No nulls
  region               ⚠ 1 nulls (2.4%)
  category             ✓ No nulls
  product              ✓ No nulls
  quantity             ✓ No nulls
  unit_price_inr       ✓ No nulls
  discount_rate        ⚠ 1 nulls (2.4%)
  status               ✓ No nulls


In [0]:
# Check for fully duplicate records (all columns identical)
dup_count = df.count() - df.dropDuplicates().count()
print(f"Total rows:            {total_rows:,}")
print(f"Distinct rows:         {df.dropDuplicates().count():,}")
print(f"Duplicate rows:        {dup_count:,}")

# Check for duplicate order_id values (possible key violation)
dup_order_ids = df.groupBy("order_id").count().filter("count > 1").count()
print(f"\nDuplicate order_ids:   {dup_order_ids:,}")
if dup_order_ids > 0:
    print("\nSample duplicate order_ids:")
    display(df.groupBy("order_id").count().filter("count > 1").orderBy(col("count").desc()).limit(10))

Total rows:            41
Distinct rows:         40
Duplicate rows:        1

Duplicate order_ids:   1

Sample duplicate order_ids:


order_id,count
O1005,2


In [0]:
# Examine unique values for categorical columns: status, category, region
from pyspark.sql.functions import desc

for col_name in ["status", "category", "region"]:
    print(f"\n{'='*50}")
    print(f"  Unique values in '{col_name}'")
    print(f"{'='*50}")
    value_counts = df.groupBy(col_name).count().orderBy(desc("count"))
    distinct_count = value_counts.count()
    print(f"Distinct {col_name}s: {distinct_count}")
    display(value_counts)


  Unique values in 'status'
Distinct statuss: 4


status,count
Delivered,36
Cancelled,2
Returned,2
delivered,1



  Unique values in 'category'
Distinct categorys: 3


category,count
Electronics,14
Home,14
Fashion,13



  Unique values in 'region'
Distinct regions: 5


region,count
North,11
South,10
West,10
East,9
null,1


In [0]:
# === INVESTIGATION: Duplicates ===
# Show ALL rows where order_id appears more than once (by full row and by order_id)
print("=== Duplicate order_id investigation: O1005 ===")
print("All rows with order_id = O1005:")
display(df.filter(col("order_id") == "O1005"))

# Are the duplicates full-row duplicates, or same order_id with different data?
dup_by_id = df.groupBy("order_id").count().filter("count > 1")
dup_ids = [r.order_id for r in dup_by_id.collect()]
print(f"\nDuplicate order_ids found: {dup_ids}")
for oid in dup_ids:
    rows = df.filter(col("order_id") == oid).collect()
    all_same = all(rows[0] == rows[i] for i in range(1, len(rows)))
    print(f"  {oid}: {len(rows)} rows, {'IDENTICAL (full-row duplicate)' if all_same else 'DIFFERENT (same ID, different data)'}")

=== Duplicate order_id investigation: O1005 ===
All rows with order_id = O1005:


order_id,order_date,region,category,product,quantity,unit_price_inr,discount_rate,status
O1005,2026-09-03,North,Home,Storage Box,4,299,0.0,Delivered
O1005,2026-09-03,North,Home,Storage Box,4,299,0.0,Delivered



Duplicate order_ids found: ['O1005']
  O1005: 2 rows, IDENTICAL (full-row duplicate)


In [0]:
# === INVESTIGATION: Missing Values ===
# Show the full rows where region or discount_rate is null
print("=== Row(s) with NULL region ===")
display(df.filter(col("region").isNull()))

print("\n=== Row(s) with NULL discount_rate ===")
display(df.filter(col("discount_rate").isNull()))

=== Row(s) with NULL region ===


order_id,order_date,region,category,product,quantity,unit_price_inr,discount_rate,status
O1008,2026-09-04,null,Home,Desk Lamp,2,1299,0.05,Delivered



=== Row(s) with NULL discount_rate ===


order_id,order_date,region,category,product,quantity,unit_price_inr,discount_rate,status
O1013,2026-09-07,North,Electronics,Wireless Mouse,3,799,null,Delivered


In [0]:
# === INVESTIGATION: Invalid Quantities ===
# Find rows with quantity <= 0 (zero or negative)
print("=== Rows with quantity <= 0 ===")
zero_qty = df.filter(col("quantity") <= 0)
zero_count = zero_qty.count()
print(f"Count: {zero_count}")
if zero_count > 0:
    display(zero_qty)
else:
    print("No zero or negative quantities found.")

# Also check for negative prices or unusual discount rates
print("\n=== Rows with unit_price_inr <= 0 ===")
bad_price = df.filter(col("unit_price_inr") <= 0)
print(f"Count: {bad_price.count()}")
if bad_price.count() > 0:
    display(bad_price)

print("\n=== Rows with discount_rate outside [0, 1] ===")
bad_disc = df.filter((col("discount_rate") < 0) | (col("discount_rate") > 1))
print(f"Count: {bad_disc.count()}")
if bad_disc.count() > 0:
    display(bad_disc)

=== Rows with quantity <= 0 ===
Count: 1


order_id,order_date,region,category,product,quantity,unit_price_inr,discount_rate,status
O1022,2026-09-11,South,Electronics,USB-C Hub,0,2499,0.2,Returned



=== Rows with unit_price_inr <= 0 ===
Count: 0

=== Rows with discount_rate outside [0, 1] ===
Count: 0


In [0]:
# === STANDARDIZE: Status labels ===
from pyspark.sql.functions import initcap, trim

# Before
df_standardized = df.withColumn(
    "status", initcap(trim(col("status")))
)

print("=== Status values BEFORE standardization ===")
df.groupBy("status").count().orderBy(desc("count")).show()

print("=== Status values AFTER standardization (initcap) ===")
df_standardized.groupBy("status").count().orderBy(desc("count")).show()

=== Status values BEFORE standardization ===
+---------+-----+
|   status|count|
+---------+-----+
|Delivered|   36|
|Cancelled|    2|
| Returned|    2|
|delivered|    1|
+---------+-----+

=== Status values AFTER standardization (initcap) ===
+---------+-----+
|   status|count|
+---------+-----+
|Delivered|   37|
|Cancelled|    2|
| Returned|    2|
+---------+-----+



In [0]:
# === BUILD CLEANED DATASET ===
# Strategy:
# 1. Drop full-row duplicate (keep first occurrence by order_id)
# 2. Standardize status casing (initcap)
# 3. Flag missing region for review (fill with 'Unknown')
# 4. Flag missing discount_rate for review (fill with 0.0)
# 5. Flag zero/negative quantity for review
# 6. Add 'cleaning_action' column documenting each action

from pyspark.sql.functions import when, lit, row_number, trim, initcap, concat, col
from pyspark.sql.window import Window

# Step 1: Deduplicate — keep first row per order_id (full-row duplicates are identical)
w = Window.partitionBy("order_id").orderBy("order_date")
df_deduped = df_standardized.withColumn("_rn", row_number().over(w)).filter("_rn = 1").drop("_rn")

# Step 2: Standardize status (already done in df_standardized, carried through)

# Step 3-5: Build cleaning_action flags
# Start with a list of actions per row
df_cleaned = df_deduped.withColumn(
    "cleaning_action",
    lit("")
)

# Flag: was this a duplicate that got removed? (already removed, so nothing to flag here)
# Flag: missing region
df_cleaned = df_cleaned.withColumn(
    "cleaning_action",
    when(col("region").isNull(), concat(col("cleaning_action"), lit("missing_region; ")))
    .otherwise(col("cleaning_action"))
)

# Flag: missing discount_rate
df_cleaned = df_cleaned.withColumn(
    "cleaning_action",
    when(col("discount_rate").isNull(), concat(col("cleaning_action"), lit("missing_discount; ")))
    .otherwise(col("cleaning_action"))
)

# Flag: invalid quantity (0 or negative)
df_cleaned = df_cleaned.withColumn(
    "cleaning_action",
    when(col("quantity") <= 0, concat(col("cleaning_action"), lit("invalid_quantity; ")))
    .otherwise(col("cleaning_action"))
)

# Flag: standardized status (was non-standard casing)
df_cleaned = df_cleaned.withColumn(
    "cleaning_action",
    when(col("status") != initcap(trim(col("status"))), concat(col("cleaning_action"), lit("status_casing_fixed; ")))
    .otherwise(col("cleaning_action"))
)

# Clean up empty action labels
df_cleaned = df_cleaned.withColumn(
    "cleaning_action",
    when(trim(col("cleaning_action")) == "", lit("none")).otherwise(col("cleaning_action"))
)

# Fill missing values
df_cleaned = df_cleaned.fillna({"region": "Unknown", "discount_rate": 0.0})

# Final report
print("=== CLEANED DATASET SUMMARY ===")
print(f"Original rows:     {df.count():,}")
print(f"After dedup:       {df_deduped.count():,}  (removed {df.count() - df_deduped.count()} duplicate)")
print(f"Final rows:        {df_cleaned.count():,}")
print(f"Columns:           {len(df_cleaned.columns)} (added 'cleaning_action')")
print(f"Missing values:    {df_cleaned.select([count(when(isnull(c), c)).alias(c) for c in df_cleaned.columns]).collect()[0].asDict()}")

print("\n=== Records flagged for review ===")
flagged = df_cleaned.filter(col("cleaning_action") != "none")
print(f"Flagged records: {flagged.count()}")
display(flagged)

=== CLEANED DATASET SUMMARY ===
Original rows:     41
After dedup:       40  (removed 1 duplicate)
Final rows:        40
Columns:           10 (added 'cleaning_action')
Missing values:    {'order_id': 0, 'order_date': 0, 'region': 0, 'category': 0, 'product': 0, 'quantity': 0, 'unit_price_inr': 0, 'discount_rate': 0, 'status': 0, 'cleaning_action': 0}

=== Records flagged for review ===
Flagged records: 3


order_id,order_date,region,category,product,quantity,unit_price_inr,discount_rate,status,cleaning_action
O1008,2026-09-04,Unknown,Home,Desk Lamp,2,1299,0.05,Delivered,missing_region;
O1013,2026-09-07,North,Electronics,Wireless Mouse,3,799,0.0,Delivered,missing_discount;
O1022,2026-09-11,South,Electronics,USB-C Hub,0,2499,0.2,Returned,invalid_quantity;


In [0]:
# === VERIFY: Cleaned data quality ===
print("=== Post-cleaning verification ===")
print(f"Total rows:            {df_cleaned.count():,}")
print(f"Distinct rows:         {df_cleaned.dropDuplicates().count():,} (should equal total)")
print(f"Duplicate order_ids:    {df_cleaned.groupBy('order_id').count().filter('count > 1').count()} (should be 0)")

print("\n--- Status distribution (post-cleaning) ---")
df_cleaned.groupBy("status").count().orderBy(desc("count")).show()

print("--- Region distribution (post-cleaning) ---")
df_cleaned.groupBy("region").count().orderBy(desc("count")).show()

print("--- discount_rate nulls (post-cleaning) ---")
print(f"  nulls: {df_cleaned.filter(col('discount_rate').isNull()).count()}")

print("--- quantity distribution (post-cleaning) ---")
df_cleaned.groupBy("quantity").count().orderBy("quantity").show()

print("\n--- Cleaning action breakdown ---")
df_cleaned.groupBy("cleaning_action").count().orderBy(desc("count")).show(truncate=False)

=== Post-cleaning verification ===
Total rows:            40
Distinct rows:         40 (should equal total)
Duplicate order_ids:    0 (should be 0)

--- Status distribution (post-cleaning) ---
+---------+-----+
|   status|count|
+---------+-----+
|Delivered|   36|
|Cancelled|    2|
| Returned|    2|
+---------+-----+

--- Region distribution (post-cleaning) ---
+-------+-----+
| region|count|
+-------+-----+
|   West|   10|
|  South|   10|
|  North|   10|
|   East|    9|
|Unknown|    1|
+-------+-----+

--- discount_rate nulls (post-cleaning) ---
  nulls: 0
--- quantity distribution (post-cleaning) ---
+--------+-----+
|quantity|count|
+--------+-----+
|       0|    1|
|       1|   16|
|       2|   15|
|       3|    5|
|       4|    2|
|       5|    1|
+--------+-----+


--- Cleaning action breakdown ---
+------------------+-----+
|cleaning_action   |count|
+------------------+-----+
|none              |37   |
|missing_region;   |1    |
|missing_discount; |1    |
|invalid_quantity; |1 

In [0]:
# === SAVE CLEANED DATASET AS A NEW TABLE ===
cleaned_table = "data_learn.default.ecommerce_sales_cleaned"

# Drop the table if it already exists (idempotent re-runs)
spark.sql(f"DROP TABLE IF EXISTS {cleaned_table}")

# Write the cleaned dataframe as a managed Delta table
df_cleaned.write.mode("overwrite").saveAsTable(cleaned_table)

# Verify the saved table
saved = spark.table(cleaned_table)
print(f"Saved to: {cleaned_table}")
print(f"Rows:     {saved.count():,}")
print(f"Columns:  {len(saved.columns)}")
print(f"\nSchema:")
saved.printSchema()
print("\nFirst 5 rows:")
display(saved.limit(5))

Saved to: data_learn.default.ecommerce_sales_cleaned
Rows:     40
Columns:  10

Schema:
root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price_inr: long (nullable = true)
 |-- discount_rate: double (nullable = true)
 |-- status: string (nullable = true)
 |-- cleaning_action: string (nullable = true)


First 5 rows:


order_id,order_date,region,category,product,quantity,unit_price_inr,discount_rate,status,cleaning_action
O1001,2026-09-01,North,Electronics,Wireless Mouse,2,799,0.05,Delivered,none
O1002,2026-09-01,South,Home,Desk Lamp,1,1299,0.1,Delivered,none
O1003,2026-09-02,West,Electronics,USB-C Hub,1,2499,0.0,Delivered,none
O1004,2026-09-02,East,Fashion,Cotton T-Shirt,3,499,0.15,Delivered,none
O1005,2026-09-03,North,Home,Storage Box,4,299,0.0,Delivered,none


## Sales Performance Metrics

Key KPIs computed from the cleaned dataset (`df_cleaned`):
- Total orders and orders by status
- Gross merchandise value (GMV) before discounts
- Net sales after discounts
- Net sales by product category
- Average order value (AOV)
- Order cancellation and return rates

In [0]:
# === SALES METRICS ===
# Add line-level GMV and net sales columns
from pyspark.sql.functions import sum as _sum, round as _round, count as _count, col, desc

metrics_df = df_cleaned.withColumn(
    "gmv", col("quantity") * col("unit_price_inr")
).withColumn(
    "net_sales", col("gmv") * (1 - col("discount_rate")))

# --- Total Orders & Orders by Status ---
print("=== Total Orders ===")
total_orders = metrics_df.count()
print(f"Total orders: {total_orders}")

print("\n=== Orders by Status ===")
status_dist = metrics_df.groupBy("status").agg(
    _count("*").alias("order_count"),
    _round((_count("*") / total_orders) * 100, 1).alias("pct_of_total")
).orderBy(desc("order_count"))
display(status_dist)

# --- GMV & Net Sales (all orders) ---
print("\n=== Gross Merchandise Value & Net Sales (all orders) ===")
overall = metrics_df.agg(
    _round(_sum("gmv"), 2).alias("gmv_total"),
    _round(_sum("net_sales"), 2).alias("net_sales_total"),
    _round(_sum("gmv") - _sum("net_sales"), 2).alias("total_discount_given")
)
display(overall)

# --- GMV & Net Sales (Delivered orders only) ---
print("\n=== GMV & Net Sales (Delivered orders only) ===")
delivered = metrics_df.filter(col("status") == "Delivered").agg(
    _round(_sum("gmv"), 2).alias("gmv_delivered"),
    _round(_sum("net_sales"), 2).alias("net_sales_delivered")
)
display(delivered)

=== Total Orders ===
Total orders: 40

=== Orders by Status ===


status,order_count,pct_of_total
Delivered,36,90.0
Cancelled,2,5.0
Returned,2,5.0



=== Gross Merchandise Value & Net Sales (all orders) ===


gmv_total,net_sales_total,total_discount_given
82726,75546.35,7179.65



=== GMV & Net Sales (Delivered orders only) ===


gmv_delivered,net_sales_delivered
77930,70850.15


In [0]:
# --- Net Sales by Product Category ---
print("=== Net Sales by Product Category (all orders) ===")
category_sales = metrics_df.groupBy("category").agg(
    _count("*").alias("order_count"),
    _round(_sum("gmv"), 2).alias("gmv"),
    _round(_sum("net_sales"), 2).alias("net_sales"),
    _round(_sum("net_sales") / _count("*"), 2).alias("avg_net_per_order")
).orderBy(desc("net_sales"))
display(category_sales)

# --- Average Order Value (AOV) ---
print("\n=== Average Order Value (AOV) ===")
# AOV = Net sales / Total orders (all statuses)
aov_all = metrics_df.agg(
    _round(_sum("net_sales") / _count("*"), 2).alias("aov_all_orders")
)

# AOV for delivered orders only
aov_delivered = metrics_df.filter(col("status") == "Delivered").agg(
    _round(_sum("net_sales") / _count("*"), 2).alias("aov_delivered_only")
)

print("AOV (all orders):       ", aov_all.collect()[0]["aov_all_orders"])
print("AOV (delivered only):   ", aov_delivered.collect()[0]["aov_delivered_only"])

# --- Order Cancellation & Return Rates ---
print("\n=== Order Cancellation & Return Rates ===")
cancelled_count = metrics_df.filter(col("status") == "Cancelled").count()
returned_count = metrics_df.filter(col("status") == "Returned").count()

cancel_rate = round((cancelled_count / total_orders) * 100, 1)
return_rate = round((returned_count / total_orders) * 100, 1)

print(f"Total orders:        {total_orders}")
print(f"Cancelled orders:   {cancelled_count}  ({cancel_rate}%)")
print(f"Returned orders:     {returned_count}  ({return_rate}%)")
print(f"Delivered orders:    {metrics_df.filter(col('status') == 'Delivered').count()}  ({round((metrics_df.filter(col('status') == 'Delivered').count() / total_orders) * 100, 1)}%)")
print(f"\nCancellation rate:   {cancel_rate}%")
print(f"Return rate:         {return_rate}%")
print(f"Combined churn rate: {round(cancel_rate + return_rate, 1)}%", )

=== Net Sales by Product Category (all orders) ===


category,order_count,gmv,net_sales,avg_net_per_order
Electronics,14,29580,28140.95,2010.07
Fashion,13,32475,28103.0,2161.77
Home,13,20671,19302.4,1484.8



=== Average Order Value (AOV) ===
AOV (all orders):        1888.66
AOV (delivered only):    1968.06

=== Order Cancellation & Return Rates ===
Total orders:        40
Cancelled orders:   2  (5.0%)
Returned orders:     2  (5.0%)
Delivered orders:    36  (90.0%)

Cancellation rate:   5.0%
Return rate:         5.0%
Combined churn rate: 10.0%


## Business Insights

Deep-dive analysis to answer:
1. Which category generates the most net sales?
2. Which category has the highest average order value?
3. Are discounts associated with larger order values?
4. Which regions or products deserve further investigation?

In [0]:
# === INSIGHT 1 & 2: Category net sales & AOV ranking ===
print("=== Category Performance Ranking ===")
cat_rank = metrics_df.groupBy("category").agg(
    _count("*").alias("orders"),
    _round(_sum("net_sales"), 2).alias("net_sales"),
    _round(_sum("net_sales") / _count("*"), 2).alias("aov"),
    _round(_sum("gmv"), 2).alias("gmv"),
    _round((_sum("gmv") - _sum("net_sales")) / _sum("gmv") * 100, 1).alias("discount_pct_of_gmv")
).orderBy(desc("net_sales"))
display(cat_rank)

top_cat_sales = cat_rank.first()["category"]
top_cat_aov = cat_rank.orderBy(desc("aov")).first()["category"]
print(f"\n  Most net sales:     {top_cat_sales}")
print(f"  Highest AOV:       {top_cat_aov}")

=== Category Performance Ranking ===


category,orders,net_sales,aov,gmv,discount_pct_of_gmv
Electronics,14,28140.95,2010.07,29580,4.9
Fashion,13,28103.0,2161.77,32475,13.5
Home,13,19302.4,1484.8,20671,6.6



  Most net sales:     Electronics
  Highest AOV:       Fashion


In [0]:
# === INSIGHT 3: Are discounts associated with larger order values? ===
from pyspark.sql.functions import when as _when, corr, avg, expr

# Bucket discount rates into tiers
disc_analysis = metrics_df.withColumn(
    "discount_tier",
    _when(col("discount_rate") == 0, "0% (no discount)")
    .when((col("discount_rate") > 0) & (col("discount_rate") <= 0.10), "1-10%")
    .when((col("discount_rate") > 0.10) & (col("discount_rate") <= 0.20), "11-20%")
    .otherwise("21%+")
)

print("=== Order Value by Discount Tier ===")
tier_stats = disc_analysis.groupBy("discount_tier").agg(
    _count("*").alias("orders"),
    _round(avg("gmv"), 2).alias("avg_gmv"),
    _round(avg("net_sales"), 2).alias("avg_net_sales"),
    _round(avg("quantity"), 1).alias("avg_qty"),
    _round(avg("unit_price_inr"), 2).alias("avg_unit_price")
).orderBy("avg_gmv")
display(tier_stats)

# Correlation between discount_rate and GMV (order value before discount)
print("\n=== Correlation: discount_rate vs GMV ===")
corr_val = disc_analysis.stat.corr("discount_rate", "gmv")
print(f"  Pearson correlation (discount_rate, gmv): {corr_val:.3f}")
if corr_val is not None:
    if abs(corr_val) < 0.3:
        print("  → Weak or no linear association")
    elif corr_val > 0.3:
        print("  → Positive: higher discounts tend to come with larger order values")
    else:
        print("  → Negative: higher discounts tend to come with smaller order values")

# Also check discount_rate vs quantity
corr_qty = disc_analysis.stat.corr("discount_rate", "quantity")
print(f"\n  Pearson correlation (discount_rate, quantity): {corr_qty:.3f}")

=== Order Value by Discount Tier ===


discount_tier,orders,avg_gmv,avg_net_sales,avg_qty,avg_unit_price
0% (no discount),14,1541.14,1541.14,1.7,1084.71
11-20%,7,2026.71,1687.03,1.9,1656.14
1-10%,18,2275.83,2092.37,1.9,1426.78
21%+,1,5998.0,4498.5,2.0,2999.0



=== Correlation: discount_rate vs GMV ===
  Pearson correlation (discount_rate, gmv): 0.321
  → Positive: higher discounts tend to come with larger order values

  Pearson correlation (discount_rate, quantity): 0.028


In [0]:
# === INSIGHT 4a: Regional performance ===
print("=== Net Sales by Region ===")
region_stats = metrics_df.groupBy("region").agg(
    _count("*").alias("orders"),
    _round(_sum("net_sales"), 2).alias("net_sales"),
    _round(_sum("net_sales") / _count("*"), 2).alias("aov"),
    _round(_sum("gmv"), 2).alias("gmv"),
    _round((_sum("gmv") - _sum("net_sales")) / _sum("gmv") * 100, 1).alias("disc_pct")
).orderBy(desc("net_sales"))
display(region_stats)

# Cancellation/return rates by region
print("\n=== Cancellation & Return Rates by Region ===")
region_status = metrics_df.groupBy("region").pivot("status").agg(_count("order_id")).fillna(0)
display(region_status)

=== Net Sales by Region ===


region,orders,net_sales,aov,gmv,disc_pct
East,9,26151.5,2905.72,28685,8.8
South,10,24043.35,2404.34,27287,11.9
West,10,12712.45,1271.25,13476,5.7
North,10,10170.95,1017.09,10680,4.8
Unknown,1,2468.1,2468.1,2598,5.0



=== Cancellation & Return Rates by Region ===


region,Cancelled,Delivered,Returned
North,0,9,1
South,1,8,1
West,1,9,0
East,0,9,0
Unknown,0,1,0


In [0]:
# === INSIGHT 4b: Product-level investigation ===
print("=== Product Performance (by net sales) ===")
product_stats = metrics_df.groupBy("product", "category").agg(
    _count("*").alias("orders"),
    _round(_sum("net_sales"), 2).alias("net_sales"),
    _round(avg("unit_price_inr"), 2).alias("avg_price"),
    _round(avg("discount_rate") * 100, 1).alias("avg_disc_pct"),
    _sum("quantity").alias("total_qty")
).orderBy(desc("net_sales"))
display(product_stats)

# Identify products with high cancellation/return rates
print("\n=== Products with Cancelled or Returned orders ===")
problem_products = metrics_df.filter(col("status").isin("Cancelled", "Returned"))
display(problem_products.select("order_id", "product", "category", "region", "quantity", "unit_price_inr", "discount_rate", "status"))

# Anomaly check: which products have the highest avg discount rate?
print("\n=== Products ranked by avg discount rate ===")
product_disc = metrics_df.groupBy("product").agg(
    _count("*").alias("orders"),
    _round(avg("discount_rate") * 100, 1).alias("avg_disc_pct"),
    _round(avg("net_sales"), 2).alias("avg_net_sales")
).orderBy(desc("avg_disc_pct"))
display(product_disc)

=== Product Performance (by net sales) ===


product,category,orders,net_sales,avg_price,avg_disc_pct,total_qty
Running Shoes,Fashion,6,20543.15,2999.0,13.3,8
USB-C Hub,Electronics,7,18992.4,2499.0,7.1,8
Desk Lamp,Home,7,14353.95,1299.0,6.4,12
Wireless Mouse,Electronics,7,9148.55,799.0,5.0,12
Cotton T-Shirt,Fashion,7,7559.85,499.0,8.6,17
Storage Box,Home,6,4948.45,299.0,2.5,17



=== Products with Cancelled or Returned orders ===


order_id,product,category,region,quantity,unit_price_inr,discount_rate,status
O1007,Wireless Mouse,Electronics,West,1,799,0.0,Cancelled
O1009,Cotton T-Shirt,Fashion,North,2,499,0.1,Returned
O1018,Running Shoes,Fashion,South,1,2999,0.0,Cancelled
O1022,USB-C Hub,Electronics,South,0,2499,0.2,Returned



=== Products ranked by avg discount rate ===


product,orders,avg_disc_pct,avg_net_sales
Running Shoes,6,13.3,3423.86
Cotton T-Shirt,7,8.6,1079.98
USB-C Hub,7,7.1,2713.2
Desk Lamp,7,6.4,2050.56
Wireless Mouse,7,5.0,1306.94
Storage Box,6,2.5,824.74


## Business Insights — Summary

### 1. Which category generates the most net sales?

| Category | Orders | Net Sales | AOV | Discount % of GMV |
| --- | --- | --- | --- | --- |
| **Electronics** | 14 | ₹28,140.95 | ₹2,010.07 | 4.9% |
| Fashion | 13 | ₹28,103.00 | ₹2,161.77 | 13.5% |
| Home | 13 | ₹19,302.40 | ₹1,484.80 | 6.6% |

**Electronics** is the top net sales category at ₹28,141 — but only by a razor-thin ₹38 margin over Fashion. Electronics achieves this through volume (14 orders, the most of any category) and low discounting (4.9% of GMV). Home lags significantly at ₹19,302.

### 2. Which category has the highest average order value?

**Fashion** has the highest AOV at **₹2,161.77** — 7.5% above Electronics (₹2,010) and 46% above Home (₹1,485). This is driven by Running Shoes (₹2,999/unit), the most expensive product in the catalog. Notably, Fashion also has the highest discount rate (13.5% of GMV), meaning its net AOV advantage would be even larger without the discounting.

### 3. Are discounts associated with larger order values?

| Discount Tier | Orders | Avg GMV | Avg Net Sales | Avg Qty | Avg Unit Price |
| --- | --- | --- | --- | --- | --- |
| 0% (no discount) | 14 | ₹1,541 | ₹1,541 | 1.7 | ₹1,085 |
| 1-10% | 18 | ₹2,276 | ₹2,092 | 1.9 | ₹1,427 |
| 11-20% | 7 | ₹2,027 | ₹1,687 | 1.9 | ₹1,656 |
| 21%+ | 1 | ₹5,998 | ₹4,499 | 2.0 | ₹2,999 |

* **Pearson correlation (discount_rate vs GMV): 0.321** — a weak-to-moderate **positive** association
* **Pearson correlation (discount_rate vs quantity): 0.028** — essentially no relationship

Discounts correlate with **higher order values but not with more units ordered**. This suggests discounts are being applied to higher-priced items (Running Shoes at 13.3% avg discount, USB-C Hub at 7.1%) rather than driving bulk purchases. The discount strategy appears to be pricing-tier-based, not volume-incentive-based.

### 4. Regions and products deserving further investigation

#### Regional Red Flags

| Region | Orders | Net Sales | AOV | Cancelled | Returned |
| --- | --- | --- | --- | --- | --- |
| **East** | 9 | ₹26,151.50 | ₹2,905.72 | 0 | 0 |
| South | 10 | ₹24,043.35 | ₹2,404.34 | 1 | 1 |
| West | 10 | ₹12,712.45 | ₹1,271.25 | 1 | 0 |
| North | 10 | ₹10,170.95 | ₹1,017.09 | 0 | 1 |

* **East** is the standout — highest AOV (₹2,906) and zero cancellations/returns, despite having fewer orders. Worth investigating whether higher-value products are concentrated here or if the region has a different customer profile.
* **South** is the only region with both a cancellation *and* a return — and it has the highest discount rate (11.9%). The 20%-discounted USB-C Hub return (O1022, qty=0) originated here.
* **North** has the lowest AOV (₹1,017) despite no cancellations — it's generating the least revenue per order. Worth investigating if this is a pricing or product-mix issue.

#### Product Red Flags

| Product | Net Sales | Avg Discount | Issue |
| --- | --- | --- | --- |
| Running Shoes | ₹20,543 | 13.3% | Highest discount rate; 1 cancellation (O1018, South) |
| USB-C Hub | ₹18,992 | 7.1% | 1 return with qty=0 (O1022, South, 20% discount) |
| Wireless Mouse | ₹9,149 | 5.0% | 1 cancellation (O1007, West, 0% discount) |
| Cotton T-Shirt | ₹7,560 | 8.6% | 1 return (O1009, North, 10% discount) |

* **Running Shoes** is the top revenue product but carries the highest discount rate (13.3%) and had a cancellation. The discounting strategy on this premium product needs ROI validation.
* **All 4 cancelled/returned orders** involved either high-value items or discounted items — no low-value, undiscounted items were returned. This may indicate quality issues with premium products or buyer's remorse on discounted high-ticket items.
* **Storage Box** (₹4,948 net, lowest) has the lowest price (₹299) and lowest discount (2.5%) — it's a volume play that isn't generating much revenue per order.

## Evidence-Based Findings, Recommendations, Limitations & Follow-up

### Three Evidence-Based Findings

**1. Electronics and Fashion are nearly tied on net sales, but for different reasons.**
Electronics leads with ₹28,140.95 in net sales (14 orders, only 4.9% discounting), while Fashion follows at ₹28,103.00 (13 orders, 13.5% discounting). Electronics wins on volume and disciplined pricing; Fashion wins on per-order value (AOV ₹2,161.77 vs ₹2,010.07) driven by its premium product (Running Shoes at ₹2,999/unit). Home trails at ₹19,302.40 — 31% below the top two — with the lowest AOV (₹1,484.80) and a mid-range discount rate (6.6%).

**2. Discounts correlate with higher order values but not with higher quantities — the strategy is pricing-tier-based, not volume-based.**
The Pearson correlation between discount_rate and GMV is 0.321 (weak-to-moderate positive), while the correlation between discount_rate and quantity is 0.028 (essentially zero). The data shows that higher-priced items receive larger discounts (Running Shoes avg 13.3%, USB-C Hub avg 7.1%) and undiscounted orders have the lowest average GMV (₹1,541). Discounts are not incentivizing bulk purchases — they are being applied to already-expensive products.

**3. The 10% combined cancellation-and-return rate is concentrated in high-value or discounted orders, with regional clustering.**
All 4 cancelled/returned orders involved either premium products (Running Shoes ₹2,999, USB-C Hub ₹2,499) or discounted items (20% on O1022, 10% on O1009). No low-value, undiscounted order was cancelled or returned. Regionally, South is the only region with both a cancellation and a return, and it has the highest discount rate (11.9%). East, despite the highest AOV (₹2,905.72), had zero cancellations or returns.

### Two Practical Recommendations

**1. Re-evaluate the discounting strategy on premium Fashion products.**
Running Shoes carries the highest average discount rate (13.3%) yet still generated a cancellation (O1018). Fashion's discount rate of 13.5% of GMV is nearly 3x Electronics' 4.9%, but Fashion's net sales only match Electronics. A controlled A/B test reducing discounts on Running Shoes by 50% would reveal whether the discount is driving conversions or simply eroding margin without incremental volume — especially since discounts show no correlation with quantity purchased.

**2. Investigate the South region's fulfilment and return process.**
South is the only region with both a cancelled and a returned order, the highest discount rate (11.9%), and the returned order (O1022) had a zero quantity with a 20% discount — suggesting a potential data-entry or fulfilment error rather than a genuine customer return. A review of South's order processing workflow, especially for discounted high-value Electronics items, could identify operational issues reducing net realised revenue.

### One Limitation of the Analysis

**The dataset contains only 40 orders over 11 days (Sep 1–11, 2026), making all findings tentative and vulnerable to small-sample noise.**
With 6 products, 4 regions, and 3 categories, each category-region-status cell averages fewer than 2 orders. The 5% cancellation rate is based on just 2 cancelled orders; one different outcome would shift the rate to 2.5% or 7.5%. Correlation coefficients (r = 0.321 for discount vs GMV) are similarly fragile — a single outlier like the ₹5,998 order at 21%+ discount disproportionately influences the result. No statistical significance testing was performed, and the findings should be treated as hypotheses rather than conclusions.

### One Follow-Up Question to Investigate with More Data

**Does the positive relationship between discount depth and order value persist at scale, and does it translate to higher net revenue per order, or does the discount cost outweigh the revenue lift?**
With a larger dataset (ideally 3–6 months of orders), we could segment products by price tier, run a regression of net_sales on discount_rate controlling for product and region fixed effects, and measure whether orders with discounts above 10% generate higher lifetime net revenue than comparable undiscounted orders. This would directly inform whether the current pricing-tier-based discount strategy is optimal or should be restructured toward volume incentives.
